In [2]:
import os
from dataclasses import dataclass
from typing import Tuple
import pytorch_lightning as pl
import pandas as pd
import torch
from IPython.display import display
from lightning.pytorch.loggers import CSVLogger
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, random_split, TensorDataset
from torchmetrics.classification import BinaryAccuracy, BinaryROC
import numpy as np
from torch.utils.data import WeightedRandomSampler
import seaborn as sn
from torchmetrics.classification import BinaryConfusionMatrix
#import shap
import glob
from tqdm import tqdm

/mnt/home/amadovic/anaconda3/envs/pytorch_lightning/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
!nvidia-smi

Mon Mar 18 13:12:06 2024       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.125.06   Driver Version: 525.125.06   CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-SXM...  On   | 00000000:01:00.0 Off |                   On |
| N/A   32C    P0    53W / 500W |     48MiB / 81920MiB |     N/A      Default |
|                               |                      |              Enabled |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA A100-SXM...  On   | 00000000:41:00.0 Off |                   On |
| N/A   

In [11]:
test_set = 5839086

In [4]:
@dataclass
class Config:
    """Configuration options for the Lightning MNIST example.

    Args:
        data_dir : The path to the directory where the MNIST dataset is stored. Defaults to the value of
            the 'PATH_DATASETS' environment variable or '.' if not set.

        save_dir : The path to the directory where the training logs will be saved. Defaults to 'logs/'.

        batch_size : The batch size to use during training. Defaults to 256 if a GPU is available,
            or 64 otherwise.

        max_epochs : The maximum number of epochs to train the model for. Defaults to 3.

        accelerator : The accelerator to use for training. Can be one of "cpu", "gpu", "tpu", "ipu", "auto".

        devices : The number of devices to use for training. Defaults to 1.

    Examples:
        This dataclass can be used to specify the configuration options for training a PyTorch Lightning model on the
        MNIST dataset. A new instance of this dataclass can be created as follows:

        >>> config = Config()

        The default values for each argument are shown in the documentation above. If desired, any of these values can be
        overridden when creating a new instance of the dataclass:

        >>> config = Config(batch_size=128, max_epochs=5)
    """

    #data_dir: str = os.environ.get("PATH_DATASETS", ".")
    save_dir: str = "logs/"
    #batch_size: int = 256 if torch.cuda.is_available() else 64
    #max_epochs: int = 3
    #accelerator: str = "auto"
    #devices: int = 1


config = Config()

In [5]:
import gc
torch.cuda.empty_cache()
gc.collect()

5

In [6]:
#device = author1_specter.device

In [15]:
author_references_full = pd.read_hdf('/mnt/home/amadovic/neural_author_disambiguator/author_references_nov22nd_v2.h5')

In [12]:
author_references = pd.read_json('labels_test.json').sample(test_set)

In [13]:
#len(author_references['@path'].unique())/len(author_references_full['@path'].unique())

In [14]:
author2_embed = np.load('author1_embed.npy')[[author_references.index]][0]#[0:test_set]
author1_embed = np.load('author2_embed.npy')[[author_references.index]][0]#0:test_set]

In [11]:
#author_references.query('label == False')

In [12]:
class PairsANDDataModule(pl.LightningDataModule):
    def __init__(self, data_dir: str = "path/to/dir", batch_size: int = 256):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.data =  TensorDataset(torch.cat([torch.unsqueeze(torch.tensor(author1_embed, dtype = torch.float32), axis = 1),
               torch.unsqueeze(torch.tensor(author2_embed, dtype = torch.float32), axis = 1)], axis = 1),
               torch.tensor(author_references.label.to_list(), dtype=torch.bool))
    def setup(self, stage: str):
        self.specter_train, self.specter_val, self.specter_test = random_split(self.data, [1600000, 199000, 1000])
        # Calculate class weights for weighted random sampler
        labels = [label.item() for _, label in self.specter_train]
        
        class_counts = torch.bincount(torch.tensor(labels, dtype=torch.int64))

        # Number of samples you want to sample
        num_samples = len(labels)

        # Weights for each sample
        weights = [1.0 / class_counts[i] for i in range(len(class_counts))]

        # List to contain the weights of each sample
        sample_weights = []

        for idx, label in enumerate(labels):
            class_weight = weights[label]
            sample_weights.append(class_weight)

        self.sampler = WeightedRandomSampler(sample_weights, num_samples)

    def train_dataloader(self):
        return DataLoader(self.specter_train, batch_size=self.batch_size, num_workers=3, shuffle = True)

    def val_dataloader(self):
        return DataLoader(self.specter_val, batch_size=self.batch_size, num_workers=3)

    def test_dataloader(self):
        return DataLoader(self.specter_test, batch_size=self.batch_size, num_workers=0)

In [13]:
import torch
from torch.utils.data import IterableDataset, DataLoader, WeightedRandomSampler
import pytorch_lightning as pl

class ChunkedDataset(IterableDataset):
    def __init__(self, data_dir, chunk_size):
        self.data_dir = data_dir
        self.chunk_size = chunk_size
        self.current_chunk = None
        self.chunk_idx = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.current_chunk is None or self.chunk_idx >= len(self.current_chunk):
            self.load_next_chunk()

        if self.chunk_idx < len(self.current_chunk):
            sample = self.current_chunk[self.chunk_idx]
            self.chunk_idx += 1
            return sample
        else:
            raise StopIteration

    def load_next_chunk(self):
        # Load the next chunk of data from the data source
        # Assuming you have author1_embed, author2_embed, and author_references.label available
        start_idx = self.chunk_idx * self.chunk_size
        end_idx = (self.chunk_idx + 1) * self.chunk_size

        chunk_author1_embed = author1_embed[start_idx:end_idx]
        chunk_author2_embed = author2_embed[start_idx:end_idx]
        chunk_labels = author_references.label[start_idx:end_idx]

        self.current_chunk = TensorDataset(
            torch.cat([torch.unsqueeze(torch.tensor(chunk_author1_embed, dtype=torch.float32), axis=1),
                       torch.unsqueeze(torch.tensor(chunk_author2_embed, dtype=torch.float32), axis=1)], axis=1),
            torch.tensor(chunk_labels.to_list(), dtype=torch.bool)
        )
        self.chunk_idx = 0

import pytorch_lightning as pl
from torch.utils.data import DataLoader

class PairsANDDataModule2(pl.LightningDataModule):
    def __init__(self, data_dir: str = "path/to/dir", batch_size: int = 1024, chunk_size: int = 100000):
        super().__init__()
        self.data_dir = data_dir
        self.batch_size = batch_size
        self.chunk_size = chunk_size

    def setup(self, stage: str):
        self.dataset = ChunkedDataset(self.data_dir, self.chunk_size)

    def train_dataloader(self):
        return DataLoader(self.dataset, batch_size=self.batch_size, num_workers=3)

    def val_dataloader(self):
        return DataLoader(self.dataset, batch_size=self.batch_size, num_workers=3)

    def test_dataloader(self):
        return DataLoader(self.dataset, batch_size=self.batch_size, num_workers=0)

In [14]:
dataset = PairsANDDataModule2()

In [15]:
!nvidia-smi

Mon Mar 18 10:00:24 2024       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.60.13    Driver Version: 525.60.13    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-SXM...  On   | 00000000:01:00.0 Off |                   On |
| N/A   32C    P0    51W / 500W |     48MiB / 81920MiB |     N/A      Default |
|                               |                      |              Enabled |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA A100-SXM...  On   | 00000000:41:00.0 Off |                   On |
| N/A   

In [16]:
dataset.prepare_data()
dataset.setup(0)
loader = dataset.test_dataloader()
for tensor,label in iter(loader):
    print(tensor,label)
    break

tensor([[[ 3.7745e-01,  4.2127e-04, -7.8619e-01,  ...,  6.6646e-01,
           3.7341e-01, -5.9386e-01],
         [ 3.7745e-01,  4.2127e-04, -7.8619e-01,  ..., -1.5898e-02,
           8.0854e-01,  2.5049e-01]],

        [[ 5.4860e-01, -2.1614e-01, -9.1515e-01,  ..., -2.7485e-01,
           2.5987e-01,  7.0117e-01],
         [ 4.6393e-01, -1.0412e-01, -8.4447e-01,  ...,  5.1544e-01,
           3.0494e-01,  7.3667e-01]],

        [[ 3.1185e-01, -2.3526e-01, -7.3958e-01,  ..., -1.2824e+00,
          -4.6865e-01, -1.7508e+00],
         [ 4.2568e-01, -1.9922e-02, -7.1119e-01,  ..., -1.8684e+00,
          -7.3564e-01, -6.3374e-02]],

        ...,

        [[ 2.4367e-01,  2.6639e-01, -7.8716e-01,  ..., -1.0141e+00,
          -7.1146e-01, -6.7602e-01],
         [ 4.8208e-01, -2.9626e-01, -9.0957e-01,  ..., -1.1099e+00,
          -1.3239e+00,  3.4336e-01]],

        [[ 3.8916e-01, -2.6702e-02, -8.8852e-01,  ..., -9.1451e-01,
           2.9928e-01, -2.2719e+00],
         [ 2.8583e-01,  1.9186e-0

In [17]:
class ANDismabiguator(pl.LightningModule): 
    
    def __init__(self,dropout_prob=0.50):
        super().__init__()
        self.l1 = nn.Sequential(nn.Linear(868, 1024),
                                nn.Tanh(),
                                nn.Dropout(dropout_prob), 
                                nn.Linear(1024, 512), 
                                nn.ReLU(), 
                                nn.Dropout(dropout_prob), 
                                nn.Linear(512, 64), 
                                nn.Sigmoid())
        self.margin = 0.5
        #self.accuracy = BinaryAccuracy(threshold = 0.5)#triplet
        self.accuracy = BinaryAccuracy(threshold = 0.5)
        self.BCELoss = nn.BCELoss() 
        self.CosineEmbeddingLoss = nn.CosineEmbeddingLoss(margin = self.margin)
        self.TripletMarginWithDistanceLoss = nn.TripletMarginWithDistanceLoss(distance_function = nn.CosineSimilarity(), margin = self.margin)
        self.trainoutputs = []
        self.outputs = []
        self.bcm = BinaryConfusionMatrix()
        
    def forward(self, x):
        return self.l1(x)
    
    def ANDloss(self,batch):
        x, y = batch
        y_hat = self(x)
        sign = torch.where(y, torch.tensor(1), torch.tensor(-1))
        cosine_sim = torch.cosine_similarity(y_hat[:,0,:] , y_hat[:,1,:])
        loss = self.CosineEmbeddingLoss(y_hat[:,0,:] , y_hat[:,1,:] ,sign)
        accuracy = self.accuracy(torch.clamp(cosine_sim, min = 0, max = 1), y)
        total_loss = loss.mean() 
        self.trainoutputs.append({'loss': loss, 'y': y, 'y_hat': y_hat, 'x': x, 'CS': cosine_sim, 'accuracy':accuracy})
        
        return accuracy, total_loss
    
    def nceloss(self, batch):
        
        temperature = 1.0
        x, y = batch
        y_hat = self(x)

        cosine_sim = torch.cosine_similarity(y_hat[:, 0, :], y_hat[:, 1, :])
        positive_indices = torch.where(y == 1)[0]
        # InfoNCE loss
        cosine_sim = cosine_sim / temperature
        nll = -cosine_sim[positive_indices] + torch.logsumexp(cosine_sim, dim=-1)
        loss = nll.mean()
        
        accuracy = self.accuracy(torch.clamp(cosine_sim, min=0, max=1), y)
        if self.training:
             self.trainoutputs.append({'loss': loss, 'y': y, 'y_hat': y_hat, 'x': x, 'CS': cosine_sim, 'accuracy':accuracy})

        return accuracy, loss


    def triplet_loss(self, batch):
        
        x, y = batch
        y_hat = self(x)
        #pdist = nn.PairwiseDistance()
        #sim = pdist(y_hat[:, 0, :], y_hat[:, 1, :])
        sim = torch.cosine_similarity(y_hat[:, 0, :], y_hat[:, 1, :])

        positive_indices = torch.where(y == 0)[0]
        negative_indices = torch.where(y == 1)[0]

        num_positive = len(positive_indices)
        num_negative = len(negative_indices)

        #if num_positive == 0 or num_negative == 0:
            # Handle the case where there are no positive or negative samples
            #return None, None

        # Ensure the same number of positive and negative embeddings
        num_triplets = min(num_positive, num_negative)
        
        scramble_indices = np.random.randint(0,num_triplets, size= num_triplets)

        # Extract embeddings for the required number of triplets
        anchor_embeddings = y_hat[positive_indices[:num_triplets]][:, 0, :]
        positive_embeddings = y_hat[positive_indices[:num_triplets]][:, 1, :]
        negative_embeddings = y_hat[scramble_indices[:num_triplets]][:, 1, :]
        #print(anchor_embeddings.shape)

        # Triplet loss formulation using self.TripletMarginWithDistanceLoss
        loss = self.TripletMarginWithDistanceLoss(anchor_embeddings, positive_embeddings, negative_embeddings)

        accuracy = self.accuracy(torch.clamp(sim, min=0, max=1), y)
        #print(accuracy)
        
        if self.training:
             self.trainoutputs.append({'loss': loss, 'y': y, 'y_hat': y_hat, 'x': x, 'CS': sim, 'accuracy':accuracy})

        return accuracy, loss
    
    def training_step(self, batch, batch_idx):
        accuracy, loss = self.nceloss(batch)
        self.log("train_loss", loss, prog_bar=True)
        self.log("train_acc", accuracy, prog_bar=True)
        
        return loss
    
    def validation_step(self, batch, batch_idx):
        x, y = batch
        y_hat = self(x)
        cosine_sim = torch.cosine_similarity(y_hat[:, 0, :], y_hat[:, 1, :])
        accuracy, loss = self.nceloss(batch)
        #print(accuracy)
        self.log("val_loss", loss, prog_bar=True)
        self.log("val_acc", accuracy, prog_bar=True)
        
        self.outputs.append({'loss': loss, 'y': y, 'y_hat':y_hat, 'CS': cosine_sim, 'x': x, 'accuracy': accuracy})
        return loss

    def on_train_epoch_end(self):
        y = []
        y_hat = []
        loss = []
        CS = []
        x = []
        accuracy = []
        for outputs in self.trainoutputs[-1:]:
            y.append(outputs['y'])
            y_hat.append(outputs['y_hat'])
            loss.append(outputs['loss'])
            CS.append(outputs['CS'])
            x.append(outputs['x'])
            accuracy.append(outputs['accuracy'])
        print(len(y))
        y = torch.stack(y).flatten(0,1)
        y_hat = torch.stack(y_hat).flatten(0,1)
        #loss = torch.stack(loss).flatten(0,1)
        CS = torch.stack(CS).flatten(0,1)
        x = torch.stack(x).flatten(0,1)
        #accuracy = torch.stack(accuracy).flatten(0,1)
        self.trainoutputs.clear()

        # Compute ROC using BinaryROC
        roc = BinaryROC()
        preds = CS  
        targets = y 
        fpr, tpr, thresholds = roc(preds, targets)

        # Calculate G-Mean for each threshold
        gmeans = torch.sqrt(tpr * (1 - fpr))

        # Find the optimal threshold
        index = torch.argmax(gmeans)
        optimal_threshold = thresholds[index]

        self.log("train_end_acc", tpr[index], prog_bar=True)
        self.log("train_end_threshold", optimal_threshold.item(), prog_bar=True)
        
    def on_validation_epoch_end(self):
        y = []
        y_hat = []
        loss = []
        CS = []
        x = []
        accuracy = []
        for outputs in self.outputs[-1:]:
            y.append(outputs['y'])
            y_hat.append(outputs['y_hat'])
            loss.append(outputs['loss'])
            CS.append(outputs['CS'])
            x.append(outputs['x'])
            accuracy.append(outputs['accuracy'])
            
        y = torch.stack(y).flatten(0,1)
        y_hat = torch.stack(y_hat).flatten(0,1)
        #loss = torch.stack(loss).flatten(0,1)
        CS = torch.stack(CS).flatten(0,1)
        x = torch.stack(x).flatten(0,1)
        #accuracy = torch.stack(accuracy).flatten(0,1)
        self.outputs.clear()

        # Compute ROC using BinaryROC
        roc = BinaryROC()
        preds = CS  
        targets = y 
        fpr, tpr, thresholds = roc(preds, targets)

        # Calculate G-Mean for eachhttps://moria-jupyterhub.egr.msu.edu/user/vicenteamado/notebooks/storage/projects/deepthought/data/neural_name_dismabiguator/pytorch_playground.ipynb# threshold
        gmeans = torch.sqrt(tpr * (1 - fpr))

        # Find the optimal threshold
        index = torch.argmax(gmeans)
        optimal_threshold = thresholds[index]
        
        self.log("val_end_acc", tpr[index], prog_bar=True)
        self.log("val_end_threshold", optimal_threshold.item(), prog_bar=True)
        
        # Collect misclassified data points
        misclassified_indices = (y != torch.where(CS < optimal_threshold, torch.tensor(0), torch.tensor(1)))  # Adjust the threshold as needed
        misclassified_samples = x[misclassified_indices]
        misclassified_labels = y[misclassified_indices]
        
        # Define the directory and file path to save misclassified samples
        save_dir = "misclassified_samples"
        os.makedirs(save_dir, exist_ok=True)  # Create the directory if it doesn't exist
        file_path = os.path.join(save_dir, f"misclassified_samples_epoch{self.current_epoch}.pt")
        
        # Save misclassified samples to a file
        torch.save({'samples': misclassified_samples, 'labels': misclassified_labels}, file_path)
        preds = torch.where(CS<optimal_threshold, torch.tensor(0), torch.tensor(1))
        #print(self.bcm(preds, y))
        
    def test_step(self, batch, batch_idx):
        accuracy, loss = self.ANDloss(batch)
        self.log("test_loss", loss, prog_bar=True)
        self.log("test_acc", accuracy, prog_bar=True)
        return loss
    
    def configure_optimizers(self):
        return torch.optim.Adamax(self.parameters(), lr = 0.001)

In [18]:
# import torch

# # Get the number of available GPUs
# num_gpus = torch.cuda.device_count()

# if num_gpus > 0:
#     print("Number of available GPUs:", num_gpus)
#     for i in range(num_gpus):
#         gpu_properties = torch.cuda.get_device_properties(i)
#         print(f"GPU {i}: {gpu_properties.name}, Memory: {gpu_properties.total_memory / (1024 ** 3):.2f}GB")
# else:
#     print("No GPUs available.")

In [19]:
!nvidia-smi

Mon Mar 18 10:00:27 2024       
+-----------------------------------------------------------------------------+
| NVIDIA-SMI 525.60.13    Driver Version: 525.60.13    CUDA Version: 12.0     |
|-------------------------------+----------------------+----------------------+
| GPU  Name        Persistence-M| Bus-Id        Disp.A | Volatile Uncorr. ECC |
| Fan  Temp  Perf  Pwr:Usage/Cap|         Memory-Usage | GPU-Util  Compute M. |
|                               |                      |               MIG M. |
|===============================+======================+======================|
|   0  NVIDIA A100-SXM...  On   | 00000000:01:00.0 Off |                   On |
| N/A   33C    P0    51W / 500W |     48MiB / 81920MiB |     N/A      Default |
|                               |                      |              Enabled |
+-------------------------------+----------------------+----------------------+
|   1  NVIDIA A100-SXM...  On   | 00000000:41:00.0 Off |                   On |
| N/A   

In [ ]:
dataset.setup(0)
train_loader = dataset.train_dataloader()
val_loader = dataset.val_dataloader()
trainer = pl.Trainer(accelerator="gpu",  devices=1, max_epochs= 100, logger=CSVLogger(save_dir=config.save_dir), log_every_n_steps=1)
model = ANDismabiguator()

trainer.fit(model, dataset)

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
IPU available: False, using: 0 IPUs
HPU available: False, using: 0 HPUs
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [MIG-0b7a68bb-6a20-5b68-ac22-1f6dc3505011,MIG-1f80db23-93f0-5b0f-8339-7d0ce6db0182,MIG-500768f5-7e46-5fc1-b007-c3c3251ca10c,MIG-e5f2972e-979e-57c4-bd1e-1ad1d24ff895]

  | Name                          | Type                          | Params
--------------------------------------------------------------------------------
0 | l1                            | Sequential                    | 1.4 M 
1 | accuracy                      | BinaryAccuracy                | 0     
2 | BCELoss                       | BCELoss                       | 0     
3 | CosineEmbeddingLoss           | CosineEmbeddingLoss           | 0     
4 | TripletMarginWithDistanceLoss | TripletMarginWithDistanceLoss | 0     
5 | bcm                           | BinaryConfusionMatrix         | 0     
-------------------------------------

Epoch 0: : 294it [00:08, 34.39it/s, v_num=88, train_loss=6.250, train_acc=0.899]
Validation: 0it [00:00, ?it/s]
Validation: 0it [00:00, ?it/s]
Validation DataLoader 0: : 0it [00:00, ?it/s]
Validation DataLoader 0: : 1it [00:00, 77.03it/s]
Validation DataLoader 0: : 2it [00:00, 74.85it/s]
Validation DataLoader 0: : 3it [00:00, 74.06it/s]
Validation DataLoader 0: : 4it [00:00, 76.45it/s]
Validation DataLoader 0: : 5it [00:00, 75.06it/s]
Validation DataLoader 0: : 6it [00:00, 75.68it/s]
Validation DataLoader 0: : 7it [00:00, 76.20it/s]
Validation DataLoader 0: : 8it [00:00, 76.58it/s]
Validation DataLoader 0: : 9it [00:00, 76.97it/s]
Validation DataLoader 0: : 10it [00:00, 77.32it/s]
Validation DataLoader 0: : 11it [00:00, 77.49it/s]
Validation DataLoader 0: : 12it [00:00, 77.75it/s]
Validation DataLoader 0: : 13it [00:00, 78.04it/s]
Validation DataLoader 0: : 14it [00:00, 78.04it/s]
Validation DataLoader 0: : 15it [00:00, 78.05it/s]
Validation DataLoader 0: : 16it [00:00, 78.09it/s]
Vali

Epoch 0: : 294it [00:16, 17.76it/s, v_num=88, train_loss=6.250, train_acc=0.899, val_loss=6.660, val_acc=0.887, val_end_acc=0.939, val_end_threshold=0.633]1
Epoch 1: : 294it [00:09, 29.83it/s, v_num=88, train_loss=6.240, train_acc=0.912, val_loss=6.660, val_acc=0.887, val_end_acc=0.939, val_end_threshold=0.633, train_end_acc=0.913, train_end_threshold=0.601]
Validation: 0it [00:00, ?it/s]
Validation: 0it [00:00, ?it/s]
Validation DataLoader 0: : 0it [00:00, ?it/s]
Validation DataLoader 0: : 1it [00:00, 63.23it/s]
Validation DataLoader 0: : 2it [00:00, 67.70it/s]
Validation DataLoader 0: : 3it [00:00, 67.67it/s]
Validation DataLoader 0: : 4it [00:00, 68.83it/s]
Validation DataLoader 0: : 5it [00:00, 70.64it/s]
Validation DataLoader 0: : 6it [00:00, 70.59it/s]
Validation DataLoader 0: : 7it [00:00, 71.92it/s]
Validation DataLoader 0: : 8it [00:00, 72.85it/s]
Validation DataLoader 0: : 9it [00:00, 72.82it/s]
Validation DataLoader 0: : 10it [00:00, 73.54it/s]
Validation DataLoader 0: : 11i

Validation DataLoader 0: : 291it [00:04, 68.00it/s]
Validation DataLoader 0: : 292it [00:04, 68.06it/s]
Validation DataLoader 0: : 293it [00:04, 68.11it/s]
Epoch 1: : 294it [00:18, 15.93it/s, v_num=88, train_loss=6.240, train_acc=0.912, val_loss=6.640, val_acc=0.907, val_end_acc=0.916, val_end_threshold=0.726, train_end_acc=0.913, train_end_threshold=0.601]
Epoch 1: : 294it [00:18, 15.92it/s, v_num=88, train_loss=6.240, train_acc=0.912, val_loss=6.640, val_acc=0.907, val_end_acc=0.916, val_end_threshold=0.726, train_end_acc=0.913, train_end_threshold=0.601]1
Epoch 2: : 294it [00:09, 30.69it/s, v_num=88, train_loss=6.240, train_acc=0.912, val_loss=6.640, val_acc=0.907, val_end_acc=0.916, val_end_threshold=0.726, train_end_acc=0.927, train_end_threshold=0.590]
Validation: 0it [00:00, ?it/s]
Validation: 0it [00:00, ?it/s]
Validation DataLoader 0: : 0it [00:00, ?it/s]
Validation DataLoader 0: : 1it [00:00, 51.37it/s]
Validation DataLoader 0: : 2it [00:00, 48.25it/s]
Validation DataLoader 0

Validation DataLoader 0: : 283it [00:04, 70.74it/s]
Validation DataLoader 0: : 284it [00:04, 70.79it/s]
Validation DataLoader 0: : 285it [00:04, 70.83it/s]
Validation DataLoader 0: : 286it [00:04, 70.88it/s]
Validation DataLoader 0: : 287it [00:04, 70.92it/s]
Validation DataLoader 0: : 288it [00:04, 70.96it/s]
Validation DataLoader 0: : 289it [00:04, 70.97it/s]
Validation DataLoader 0: : 290it [00:04, 71.01it/s]
Validation DataLoader 0: : 291it [00:04, 71.05it/s]
Validation DataLoader 0: : 292it [00:04, 71.15it/s]
Validation DataLoader 0: : 293it [00:04, 71.19it/s]
Epoch 2: : 294it [00:17, 16.84it/s, v_num=88, train_loss=6.240, train_acc=0.912, val_loss=6.640, val_acc=0.915, val_end_acc=0.956, val_end_threshold=0.617, train_end_acc=0.927, train_end_threshold=0.590]
Epoch 2: : 294it [00:17, 16.84it/s, v_num=88, train_loss=6.240, train_acc=0.912, val_loss=6.640, val_acc=0.915, val_end_acc=0.956, val_end_threshold=0.617, train_end_acc=0.927, train_end_threshold=0.590]1
Epoch 3: : 294it [0

...

Validation DataLoader 0: : 219it [00:03, 69.35it/s]
Validation DataLoader 0: : 220it [00:03, 69.35it/s]
Validation DataLoader 0: : 221it [00:03, 69.38it/s]
Validation DataLoader 0: : 222it [00:03, 69.42it/s]
Validation DataLoader 0: : 223it [00:03, 69.46it/s]
Validation DataLoader 0: : 224it [00:03, 69.48it/s]
Validation DataLoader 0: : 225it [00:03, 69.51it/s]
Validation DataLoader 0: : 226it [00:03, 69.52it/s]
Validation DataLoader 0: : 227it [00:03, 69.56it/s]
Validation DataLoader 0: : 228it [00:03, 69.58it/s]
Validation DataLoader 0: : 229it [00:03, 69.59it/s]
Validation DataLoader 0: : 230it [00:03, 69.60it/s]
Validation DataLoader 0: : 231it [00:03, 69.63it/s]
Validation DataLoader 0: : 232it [00:03, 69.66it/s]
Validation DataLoader 0: : 233it [00:03, 69.69it/s]
Validation DataLoader 0: : 234it [00:03, 69.70it/s]
Validation DataLoader 0: : 235it [00:03, 69.74it/s]
Validation DataLoader 0: : 236it [00:03, 69.77it/s]
Validation DataLoader 0: : 237it [00:03, 69.79it/s]
Validation D

Validation DataLoader 0: : 211it [00:03, 67.95it/s]
Validation DataLoader 0: : 212it [00:03, 67.99it/s]
Validation DataLoader 0: : 213it [00:03, 67.95it/s]
Validation DataLoader 0: : 214it [00:03, 67.97it/s]
Validation DataLoader 0: : 215it [00:03, 67.97it/s]
Validation DataLoader 0: : 216it [00:03, 67.98it/s]
Validation DataLoader 0: : 217it [00:03, 68.01it/s]
Validation DataLoader 0: : 218it [00:03, 68.06it/s]
Validation DataLoader 0: : 219it [00:03, 68.08it/s]
Validation DataLoader 0: : 220it [00:03, 68.09it/s]
Validation DataLoader 0: : 221it [00:03, 68.11it/s]
Validation DataLoader 0: : 222it [00:03, 68.15it/s]
Validation DataLoader 0: : 223it [00:03, 68.17it/s]
Validation DataLoader 0: : 224it [00:03, 68.17it/s]
Validation DataLoader 0: : 225it [00:03, 68.19it/s]
Validation DataLoader 0: : 226it [00:03, 68.23it/s]
Validation DataLoader 0: : 227it [00:03, 68.27it/s]
Validation DataLoader 0: : 228it [00:03, 68.31it/s]
Validation DataLoader 0: : 229it [00:03, 68.33it/s]
Validation D

Validation DataLoader 0: : 203it [00:03, 67.26it/s]
Validation DataLoader 0: : 204it [00:03, 67.30it/s]
Validation DataLoader 0: : 205it [00:03, 67.35it/s]
Validation DataLoader 0: : 206it [00:03, 67.40it/s]
Validation DataLoader 0: : 207it [00:03, 67.45it/s]
Validation DataLoader 0: : 208it [00:03, 67.50it/s]
Validation DataLoader 0: : 209it [00:03, 67.51it/s]
Validation DataLoader 0: : 210it [00:03, 67.55it/s]
Validation DataLoader 0: : 211it [00:03, 67.59it/s]
Validation DataLoader 0: : 212it [00:03, 67.60it/s]
Validation DataLoader 0: : 213it [00:03, 67.63it/s]
Validation DataLoader 0: : 214it [00:03, 67.67it/s]
Validation DataLoader 0: : 215it [00:03, 67.71it/s]
Validation DataLoader 0: : 216it [00:03, 67.75it/s]
Validation DataLoader 0: : 217it [00:03, 67.80it/s]
Validation DataLoader 0: : 218it [00:03, 67.84it/s]
Validation DataLoader 0: : 219it [00:03, 67.87it/s]
Validation DataLoader 0: : 220it [00:03, 67.87it/s]
Validation DataLoader 0: : 221it [00:03, 67.89it/s]
Validation D

Validation DataLoader 0: : 195it [00:02, 68.85it/s]
Validation DataLoader 0: : 196it [00:02, 68.92it/s]
Validation DataLoader 0: : 197it [00:02, 68.97it/s]
Validation DataLoader 0: : 198it [00:02, 69.03it/s]
Validation DataLoader 0: : 199it [00:02, 69.08it/s]
Validation DataLoader 0: : 200it [00:02, 69.12it/s]
Validation DataLoader 0: : 201it [00:02, 69.16it/s]
Validation DataLoader 0: : 202it [00:02, 69.12it/s]
Validation DataLoader 0: : 203it [00:02, 69.16it/s]
Validation DataLoader 0: : 204it [00:02, 69.20it/s]
Validation DataLoader 0: : 205it [00:02, 69.25it/s]
Validation DataLoader 0: : 206it [00:02, 69.26it/s]
Validation DataLoader 0: : 207it [00:02, 69.25it/s]
Validation DataLoader 0: : 208it [00:03, 69.30it/s]
Validation DataLoader 0: : 209it [00:03, 69.32it/s]
Validation DataLoader 0: : 210it [00:03, 69.36it/s]
Validation DataLoader 0: : 211it [00:03, 69.40it/s]
Validation DataLoader 0: : 212it [00:03, 69.45it/s]
Validation DataLoader 0: : 213it [00:03, 69.50it/s]
Validation D

Validation DataLoader 0: : 188it [00:03, 47.39it/s]
Validation DataLoader 0: : 189it [00:03, 47.49it/s]
Validation DataLoader 0: : 190it [00:03, 47.56it/s]
Validation DataLoader 0: : 191it [00:04, 44.73it/s]
Validation DataLoader 0: : 192it [00:04, 44.82it/s]
Validation DataLoader 0: : 193it [00:04, 44.90it/s]
Validation DataLoader 0: : 194it [00:04, 44.95it/s]
Validation DataLoader 0: : 195it [00:04, 45.03it/s]
Validation DataLoader 0: : 196it [00:04, 45.11it/s]
Validation DataLoader 0: : 197it [00:04, 43.96it/s]
Validation DataLoader 0: : 198it [00:04, 44.05it/s]
Validation DataLoader 0: : 199it [00:04, 44.14it/s]
Validation DataLoader 0: : 200it [00:04, 44.24it/s]
Validation DataLoader 0: : 201it [00:04, 44.32it/s]
Validation DataLoader 0: : 202it [00:04, 44.41it/s]
Validation DataLoader 0: : 203it [00:04, 44.50it/s]
Validation DataLoader 0: : 204it [00:04, 44.59it/s]
Validation DataLoader 0: : 205it [00:04, 44.68it/s]
Validation DataLoader 0: : 206it [00:04, 44.77it/s]
Validation D

In [4]:
# Read in the training metrics from the CSV file generated by the logger
#metrics = pd.read_csv(f"{trainer.logger.log_dir}/metrics.csv")'logs/lightning_logs/version_2/metrics.csv'
metrics = pd.read_csv('logs/lightning_logs/version_88/metrics.csv')
# Remove the "step" column, which is not needed for our analysis
del metrics["step"]

# Set the epoch column as the index, for easier plotting
metrics.set_index("epoch", inplace=True)

# Display the first few rows of the metrics table, excluding any columns with all NaN values
#display(metrics.dropna(axis=1, how="all").head())

# Create a line plot of the training metrics using Seaborn
sn.relplot(data=metrics[['train_loss', 'val_loss']], kind="line")

EmptyDataError: No columns to parse from file

In [6]:
# Read in the training metrics from the CSV file generated by the logger
#metrics = pd.read_csv(f"{trainer.logger.log_dir}/metrics.csv")'logs/lightning_logs/version_2/metrics.csv'
metrics = pd.read_csv('logs/lightning_logs/version_88/metrics.csv')
# Remove the "step" column, which is not needed for our analysis
del metrics["step"]

# Set the epoch column as the index, for easier plotting
metrics.set_index("epoch", inplace=True)

# Display the first few rows of the metrics table, excluding any columns with all NaN values
#display(metrics.dropna(axis=1, how="all").head())

# Create a line plot of the training metrics using Seaborn
sn.relplot(data=metrics[['train_end_acc', 'val_end_acc']], kind="line")

EmptyDataError: No columns to parse from file

In [ ]:
#dataset.prepare_data()
dataset.setup(0)
loader = dataset.test_dataloader()
for tensor, label in iter(loader):
     print(tensor,label)

In [ ]:
tensor.shape

In [ ]:
tensor[:200,0,:]

In [ ]:
shap_explainer = shap.DeepExplainer(model,tensor[:200,0,:])
shap_values = shap_explainer.shap_values(tensor[201:500,1,:])

In [ ]:
tensor[201,1,:].shape

In [ ]:
len(shap_values[0:10])

In [ ]:
shap.summary_plot(shap_values[0:10], plot_type = 'bar')

In [7]:
test_incorrect = torch.load('misclassified_samples/misclassified_samples_epoch39.pt', map_location='cpu')

In [8]:
test_incorrect['samples'].shape#[0].shape

torch.Size([12, 2, 868])

In [9]:
incorrect_samples = test_incorrect['samples']

In [16]:
full_samples =  TensorDataset(torch.cat([torch.unsqueeze(torch.tensor(author1_embed, dtype = torch.float32), axis = 1),
               torch.unsqueeze(torch.tensor(author2_embed, dtype = torch.float32), axis = 1)], axis = 1),
               torch.tensor(author_references.label.to_list(), dtype=torch.bool)).tensors[0]

In [17]:
incorrect_np = incorrect_samples.numpy()
full_np = full_samples.numpy()

# Reshape incorrect_samples to broadcast with full_samples
#incorrect_np_reshaped = incorrect_np[:, np.newaxis, :]
incorrect_np_reshaped = incorrect_np

In [18]:
incorrect_np.shape, full_np.shape

((12, 2, 868), (5839086, 2, 868))

In [21]:
are_equal_list = []

# Assuming incorrect_np and full_np are your arrays
num_examples_to_compare = 12  # Number of examples from incorrect_np to compare

# Select a subset of examples from incorrect_np
subset_incorrect_np = incorrect_np[:num_examples_to_compare]

In [22]:
subset_incorrect_np.shape

(12, 2, 868)

In [23]:
are_equal_list = []

# Assuming incorrect_np and full_np are your arrays
num_examples_to_compare = 12  # Number of examples from incorrect_np to compare

# Select a subset of examples from incorrect_np
subset_incorrect_np = incorrect_np[:num_examples_to_compare]

# Use tqdm to iterate over each element in full_np with a progress bar
for element in tqdm(full_np, desc="Comparing elements"):
    # Initialize a flag to indicate if a match is found
    match_found = False
    # Iterate over each example in the subset
    for example in subset_incorrect_np:
        # Perform element-wise equality comparison with the current example
        equality_comparison = torch.eq(torch.tensor(example), torch.tensor(element))
        # Check if all elements along the last dimension are equal
        are_equal = torch.all(equality_comparison, dim=-1)
        # If all elements are equal, append True and break out of the loop
        if torch.all(are_equal):
            are_equal_list.append(True)
            match_found = True
            break
    # If no match is found, append False
    if not match_found:
        are_equal_list.append(False)

Comparing elements: 100%|██████████| 5839086/5839086 [28:58<00:00, 3358.92it/s]


In [24]:
are_equal_all = torch.tensor(are_equal_list)

In [21]:
#author_references[:2324308][are_equal_list].shape

(2, 12)

In [25]:
# Assuming embeddings_data is your array of embeddings with shape (5000, 50)
# and new_rows is the DataFrame containing the pairs with indexes

# Function to get embeddings for a given index
#def get_embedding(index):
    #return embeddings_data[index]


# Iterate over the rows in new_rows DataFrame
for _, row in author_references[are_equal_list].iterrows():
    # Get embeddings for index1 and index2
    print(row['abstract'])
    print(row['author'])
    print(row['aff'])
    print(row['abstract2'])
    print(row['author2'])
    print(row['aff2'])
    print(row['label'])
    #if _ > 10:
        #break


We apply the nonperturbative optimized linear δ expansion method to the O(N) scalar field model in three dimensions to determine the transition temperature of a dilute homogeneous Bose gas. Our results show that the shift of the transition temperature ΔT<SUB>c</SUB>/T<SUB>c</SUB> of the interacting model, compared with the ideal-gas transition temperature, really behaves as γan<SUP>1/3</SUP> where a is the s-wave scattering length and n is the number density. For N=2 our calculations yield the value γ=3.059.
Ramos, Rudnei O.
Department of Physics and Astronomy, Dartmouth College, Hanover, New Hampshire 03755-3528
The major issue related to line width roughness (LWR) is the significant LWR of the photoresist patterns printed by 193-nm lithography that is partially transferred into the gate stack during the subsequent plasma etching steps. The strategy used today to overcome this issue is to apply postlithography treatments to reduce photoresist pattern LWR before transfer. In this artic